In [1]:

import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd 
from statsmodels.tsa.seasonal import seasonal_decompose 
from prophet import Prophet 
import plotly.graph_objects as go 
from scipy import stats




data = pd.read_csv('timeseries_hse_teducation_df.csv', delimiter=',')

# Первое задание

In [3]:
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

In [4]:
def analyze_series(data, ts_name, visualize=False):

    df = data[data['ts_name'] == ts_name].copy()
    df['dttm_30'] = pd.to_datetime(df['dttm_30'])

    #Дневная сезонность 
    intraday = df[['dttm_30', 'y']].rename(columns={'dttm_30': 'ds'})
    intraday = intraday.sort_values('ds')

    model_intraday = Prophet(
        daily_seasonality=True,
        weekly_seasonality=False,
        yearly_seasonality=False,
        seasonality_mode='multiplicative'
    )

    model_intraday.fit(intraday)
    forecast_intraday = model_intraday.predict(intraday)

    daily_amp = forecast_intraday['daily'].max() - forecast_intraday['daily'].min()
    level_intraday = forecast_intraday['yhat'].mean()
    daily_flag = int(daily_amp / level_intraday > 0.05)

    
    daily = (
        df.assign(ds=df['dttm_30'].dt.date)
          .groupby('ds', as_index=False)['y']
          .sum()
    )
    daily['ds'] = pd.to_datetime(daily['ds'])

    
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode='multiplicative'
    )

    model.fit(daily)
    forecast = model.predict(daily)

    daily['yhat'] = forecast['yhat']
    daily['residuals'] = daily['y'] - daily['yhat']

    
    clean = remove_outliers_iqr(daily, 'residuals').copy()

    
    clean['day_of_month'] = clean['ds'].dt.day

    
    month_pattern = (
        clean.groupby('day_of_month')['residuals']
        .mean()
        .reset_index()
    )

   
    groups = [
        group['residuals'].values
        for _, group in clean.groupby('day_of_month')
        if len(group) > 1
    ]

    if len(groups) > 2:
        f_stat, p_value = stats.f_oneway(*groups)
        monthly_flag = int(p_value < 0.05)
    else:
        monthly_flag = 0
        p_value = 1

    
    weekly_amp = forecast['weekly'].max() - forecast['weekly'].min()
    level = forecast['yhat'].mean()
    weekly_flag = int(weekly_amp / level > 0.05)

    yearly_amp = forecast['yearly'].max() - forecast['yearly'].min()
    yearly_flag = int(yearly_amp / level > 0.05)

    
    if visualize:
        import plotly.graph_objects as go

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=daily['ds'],
            y=daily['residuals'],
            mode='lines',
            name='residuals'
        ))
        fig.update_layout(title=f"Residuals: {ts_name}")
        fig.show()

        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=month_pattern['day_of_month'],
            y=month_pattern['residuals'],
            mode='lines',
            name='mean residual'
        ))
        fig2.update_layout(title=f"Month pattern (clean residuals): {ts_name}")
        fig2.show()

    return {
        'ts_name': ts_name,
        'daily_seasonality': daily_flag,
        'weekly_seasonality': weekly_flag,
        'monthly_seasonality': monthly_flag,
        'yearly_seasonality': yearly_flag,
        'month_pattern': month_pattern  
    }

In [5]:
results = []
all_month_patterns = []

for ts in data['ts_name'].unique():
    res = analyze_series(data, ts, visualize=True)
    results.append(res)
    mp = res['month_pattern'].copy()
    mp['ts_name'] = ts
    all_month_patterns.append(mp)

summary_table = pd.DataFrame(results)


for col in ['daily_seasonality', 'weekly_seasonality',
            'monthly_seasonality', 'yearly_seasonality']:
    summary_table[col] = summary_table[col].map({1: 'есть', 0: 'нет'})

summary_table


17:27:13 - cmdstanpy - INFO - Chain [1] start processing
17:27:31 - cmdstanpy - INFO - Chain [1] done processing
17:27:37 - cmdstanpy - INFO - Chain [1] start processing
17:27:37 - cmdstanpy - INFO - Chain [1] done processing


17:27:41 - cmdstanpy - INFO - Chain [1] start processing
17:28:07 - cmdstanpy - INFO - Chain [1] done processing
17:28:13 - cmdstanpy - INFO - Chain [1] start processing
17:28:13 - cmdstanpy - INFO - Chain [1] done processing


17:28:15 - cmdstanpy - INFO - Chain [1] start processing
17:28:37 - cmdstanpy - INFO - Chain [1] done processing
17:28:43 - cmdstanpy - INFO - Chain [1] start processing
17:28:43 - cmdstanpy - INFO - Chain [1] done processing


17:28:45 - cmdstanpy - INFO - Chain [1] start processing
17:29:02 - cmdstanpy - INFO - Chain [1] done processing
17:29:08 - cmdstanpy - INFO - Chain [1] start processing
17:29:08 - cmdstanpy - INFO - Chain [1] done processing


17:29:10 - cmdstanpy - INFO - Chain [1] start processing
17:29:31 - cmdstanpy - INFO - Chain [1] done processing
17:29:38 - cmdstanpy - INFO - Chain [1] start processing
17:29:38 - cmdstanpy - INFO - Chain [1] done processing


...

17:31:29 - cmdstanpy - INFO - Chain [1] start processing
17:31:47 - cmdstanpy - INFO - Chain [1] done processing
17:31:52 - cmdstanpy - INFO - Chain [1] start processing
17:31:52 - cmdstanpy - INFO - Chain [1] done processing


17:31:54 - cmdstanpy - INFO - Chain [1] start processing
17:32:04 - cmdstanpy - INFO - Chain [1] done processing
17:32:09 - cmdstanpy - INFO - Chain [1] start processing
17:32:09 - cmdstanpy - INFO - Chain [1] done processing


17:32:10 - cmdstanpy - INFO - Chain [1] start processing
17:32:27 - cmdstanpy - INFO - Chain [1] done processing
17:32:37 - cmdstanpy - INFO - Chain [1] start processing
17:32:38 - cmdstanpy - INFO - Chain [1] done processing


17:32:40 - cmdstanpy - INFO - Chain [1] start processing
17:32:48 - cmdstanpy - INFO - Chain [1] done processing
17:32:54 - cmdstanpy - INFO - Chain [1] start processing
17:32:54 - cmdstanpy - INFO - Chain [1] done processing


,ts_name,daily_seasonality,weekly_seasonality,monthly_seasonality,yearly_seasonality,month_pattern
0,ts_name_0,нет,нет,есть,нет,day_of_month residuals 0 1...
1,ts_name_1,нет,нет,есть,нет,day_of_month residuals 0 1...
2,ts_name_2,нет,нет,есть,нет,day_of_month residuals 0 1 ...
3,ts_name_3,нет,нет,есть,нет,day_of_month residuals 0 1 ...
4,ts_name_4,нет,нет,нет,нет,day_of_month residuals 0 1 ...
5,ts_name_5,нет,нет,нет,нет,day_of_month residuals 0 1...
6,ts_name_6,нет,нет,есть,нет,day_of_month residuals 0 1...
7,ts_name_7,нет,нет,есть,нет,day_of_month residuals 0 1...
8,ts_name_8,есть,нет,нет,нет,day_of_month residuals 0 1 -...
9,ts_name_9,есть,нет,нет,нет,day_of_month residuals 0 1 ...


In [24]:
import plotly.graph_objects as go
import pandas as pd

all_month_patterns_df = pd.concat(all_month_patterns)

fig = go.Figure()

for ts in all_month_patterns_df['ts_name'].unique():
    temp = all_month_patterns_df[all_month_patterns_df['ts_name'] == ts]

    fig.add_trace(go.Scatter(
        x=temp['day_of_month'],
        y=temp['residuals'],
        mode='lines',
        name=ts
    ))

fig.update_layout(
    title='Месячные паттерны (средние очищенные остатки)',
    xaxis_title='День месяца',
    yaxis_title='Средний residual',
    template='plotly_white'
)

fig.show()

In [64]:
import pandas as pd



data = pd.read_csv('timeseries_hse_teducation_df.csv', delimiter=',')

# Перестановочный тест

## Подготовка профилей

In [3]:
import holidays
import numpy as np

def build_daily_profiles(ts_df):

    df = ts_df.rename(columns={'dttm_30': 'ds', 'y': 'y'})[['ds','y']].copy()

    df['ds'] = pd.to_datetime(df['ds'])
    df['date'] = df['ds'].dt.date

    # --- убрать NaN в исходном ряду
    df = df.dropna(subset=['y'])

    # --- статистики дня
    daily_stats = df.groupby('date')['y'].agg(['mean','std'])
    df = df.join(daily_stats, on='date')

    # --- защита от std = 0
    df['std'] = df['std'].replace(0, np.nan)

    # z-преобразование
    df['load_z'] = (df['y'] - df['mean']) / df['std']

    # если появились NaN после нормализации
    df['load_z'] = df['load_z'].fillna(0)

    # --- только будни
    df['weekday'] = df['ds'].dt.weekday
    df = df[df['weekday'] < 5]

    # --- убрать праздники
    ru_holidays = holidays.Russia(years=df['ds'].dt.year.unique())
    df = df[~df['date'].isin(ru_holidays)]

    df['time'] = df['ds'].dt.time
    df['month'] = df['ds'].dt.month
    df['date'] = df['ds'].dt.date

    profiles = df.pivot_table(
        index=['date','month'],
        columns='time',
        values='load_z'
    )

    # --- если есть пропуски во временных точках
    profiles = profiles.fillna(0)

    # --- массивы для теста
    X = profiles.values

    # финальная защита
    X = np.nan_to_num(X)

    months = profiles.index.get_level_values('month').values

    return X, months

## Статистика теста

In [4]:
from scipy.spatial.distance import pdist, squareform
import numpy as np


def compute_T_fast(D, labels):

    labels = np.array(labels)

    within = []
    between = []

    n = len(labels)

    for i in range(n):
        for j in range(i+1, n):

            if labels[i] == labels[j]:
                within.append(D[i, j])
            else:
                between.append(D[i, j])

    if len(within) == 0 or len(between) == 0:
        return np.nan

    return np.mean(between) - np.mean(within)

## Перестановочный тест

In [5]:
def permutation_test_fast(X, labels, n_perm=2000):

    labels = np.array(labels).copy()

    from scipy.spatial.distance import pdist, squareform
    D = squareform(pdist(X))

    T_obs = compute_T_fast(D, labels)

    T_perm = np.zeros(n_perm)

    for i in range(n_perm):

        shuffled = np.random.permutation(labels)

        T_perm[i] = compute_T_fast(D, shuffled)

    T_perm = T_perm[~np.isnan(T_perm)]

    p_value = np.mean(T_perm >= T_obs)

    return T_obs, T_perm, p_value

## Запуск для 12 рядов

In [6]:
import plotly.graph_objects as go
import numpy as np

results = []
ts_list = data['ts_name'].unique()

for i, ts in enumerate(ts_list):

    print(f"{i+1}/{12} Processing {ts}")

    ts_df = data[data['ts_name'] == ts].copy()

    X, months = build_daily_profiles(ts_df)

    T_obs, T_perm, p_value = permutation_test_fast(X, months, n_perm=2000)

    results.append({
        'ts_name': ts,
        'T_obs': T_obs,
        'p_value': p_value,
        'n_days': len(X)
    })


    fig = go.Figure()

    fig.add_trace(
        go.Histogram(
            x=T_perm,
            nbinsx=50,
            name='Permutation T',
            marker_color='lightblue',
            opacity=0.75
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[T_obs, T_obs],
            y=[0, max(np.histogram(T_perm, bins=50)[0])],
            mode='lines',
            line=dict(color='red', width=3),
            name='Observed T'
        )
    )

    fig.update_layout(
        title=f"Permutation Test Distribution — {ts}",
        xaxis_title="T statistic",
        yaxis_title="Count",
        bargap=0.2,
        template="plotly_white",
        legend=dict(x=0.7, y=0.95)
    )

    fig.show()

1/12 Processing ts_name_0


2/12 Processing ts_name_1


3/12 Processing ts_name_2


4/12 Processing ts_name_3


5/12 Processing ts_name_4


...

8/12 Processing ts_name_7


9/12 Processing ts_name_8


10/12 Processing ts_name_9


11/12 Processing ts_name_10


12/12 Processing ts_name_11


In [7]:
results_df = pd.DataFrame(results).sort_values('p_value')

from statsmodels.stats.multitest import multipletests
results_df['p_adj'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

print(results_df)

       ts_name     T_obs  p_value  n_days   p_adj
0    ts_name_0  0.063892   0.0000     731  0.0000
1    ts_name_1  0.067194   0.0000     731  0.0000
2    ts_name_2  0.021536   0.0000     731  0.0000
3    ts_name_3  0.164138   0.0000     731  0.0000
4    ts_name_4  0.091446   0.0000     731  0.0000
5    ts_name_5  0.127545   0.0000     731  0.0000
6    ts_name_6  0.059602   0.0000     731  0.0000
7    ts_name_7  0.071191   0.0000     731  0.0000
8    ts_name_8  0.036342   0.0000     731  0.0000
9    ts_name_9  0.036084   0.0000     673  0.0000
10  ts_name_10  0.077849   0.0000     673  0.0000
11  ts_name_11  0.015684   0.0115     583  0.0115


# Статистические тесты для выявления сезонности во временных рядах

Во временных рядах наблюдения часто **зависят от предыдущих значений**, что нарушает предпосылку независимости, характерную для классических статистических тестов. Поэтому для анализа временных рядов используются специализированные методы, учитывающие **автокорреляцию и временную структуру данных**.

Для исследования сезонности применяются несколько статистических тестов, позволяющих определить наличие сезонных зависимостей, сезонной нестационарности и стабильности сезонного компонента.

---

# 1. HEGY test (Hylleberg–Engle–Granger–Yoo)

HEGY-тест используется для выявления **сезонных единичных корней** во временных рядах.


## Основная идея теста

HEGY определяет, присутствует ли **единичный корень на сезонных частотах**.  

Если сезонный единичный корень существует, сезонная структура временного ряда является **нестационарной** и требует сезонного дифференцирования.

Примеры сезонных периодов:

| Тип данных | Сезонный период |
|---|---|
| Месячные данные | 12 |
| Квартальные данные | 4 |
| Дневные данные | 7 |

В отличие от стандартных тестов стационарности, HEGY проверяет единичные корни не только при нулевой частоте (долгосрочный тренд), но и при **сезонных частотах**.

## Гипотезы теста

**H₀:** присутствует сезонный единичный корень (ряд сезонно нестационарен)  
**H₁:** сезонный единичный корень отсутствует

## Интерпретация результатов

| Результат | Вывод |
|---|---|
Нулевая гипотеза не отклоняется | присутствует сезонный единичный корень |
Нулевая гипотеза отклоняется | сезонность стационарна |

## Области применения

HEGY-тест часто используется в:

- макроэкономике
- анализе продаж
- финансовых временных рядах
- анализе потребления энергии

---

# 2. Canova–Hansen test

Тест Canova–Hansen используется для проверки **стабильности сезонных колебаний** во временных рядах.


## Основная идея теста

Тест проверяет, **остается ли сезонная структура постоянной во времени**.

Это важно, поскольку сезонные паттерны могут изменяться из-за:

- экономических кризисов
- изменений поведения потребителей
- технологических изменений
- климатических факторов

## Гипотезы теста

**H₀:** сезонность стабильна во времени  
**H₁:** сезонность изменяется

## Интерпретация результатов

| Результат | Вывод |
|---|---|
p-value > 0.05 | сезонность стабильна |
p-value < 0.05 | сезонность изменяется |

## Практическое значение

Результаты теста помогают определить:

- можно ли использовать **сезонное дифференцирование**
- подходит ли модель **SARIMA**
- требуется ли использование моделей со **структурными изменениями**

---

# 3. Сезонный тест Ljung–Box

Тест Ljung–Box применяется для проверки **наличия автокорреляции** во временном ряду.


## Основная идея теста

Тест проверяет, существует ли **зависимость между значениями ряда на разных лагах**.

Гипотезы теста:

**H₀:** автокорреляция отсутствует  
**H₁:** автокорреляция присутствует

## Сезонная версия теста

Для анализа сезонности рассматриваются **лаги, кратные сезонному периоду**.

Примеры:

| Тип данных | Сезонный лаг |
|---|---|
Месячные данные | 12 |
Квартальные данные | 4 |
Дневные данные | 7 |

Если автокорреляция на этих лагах оказывается статистически значимой, это указывает на **наличие сезонной структуры**.

## Интерпретация результатов

| Результат | Вывод |
|---|---|
p-value > 0.05 | автокорреляция отсутствует |
p-value < 0.05 | автокорреляция присутствует |

Если значимая автокорреляция наблюдается **на сезонных лагах**, можно сделать вывод о наличии **сезонных зависимостей**.

---

# Сравнение тестов сезонности

| Тест | Что проверяет | Основная цель |
|---|---|---|
HEGY | сезонные единичные корни | выявление сезонной нестационарности |
Canova–Hansen | стабильность сезонности | проверка изменения сезонной структуры |
Ljung–Box | автокорреляцию | выявление сезонных зависимостей |

---

# Использование тестов в анализе временных рядов

При исследовании сезонности обычно применяется следующая последовательность анализа:

1. Проверка **стационарности временного ряда**
   - ADF тест
   - KPSS тест

2. Проверка **автокорреляции**
   - Ljung–Box тест

3. Анализ **сезонной структуры**
   - HEGY test
   - Canova–Hansen test

4. Построение модели временного ряда


---

# Вывод

Для анализа сезонности временных рядов используются специализированные статистические тесты, учитывающие временную зависимость данных. HEGY-тест позволяет выявлять сезонные единичные корни, тест Canova–Hansen проверяет стабильность сезонных колебаний, а тест Ljung–Box используется для выявления автокорреляции на сезонных лагах. Совместное применение этих методов позволяет более точно определить характер сезонной структуры временного ряда.